# Notebook Base — Desafío de Imágenes

Plantilla del equipo, sobre Keras/TensorFlow para mantener el stack de la
cátedra. El recorrido es el mismo que en tabular: leer la métrica, auditar,
buscar atajos, modelar con checkpoint, decidir según el costo, enviar,
documentar.

Entorno de ejecución → Cambiar tipo de entorno → **GPU**.

## Entorno

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/AcostaAlex10/hackathon-kit-Acosta-Borges-Pelinski.git"
RAMA = "herramientas"

if not os.path.isdir("hackathon-kit-Acosta-Borges-Pelinski"):
    subprocess.run(["git", "clone", "-b", RAMA, REPO, "hackathon-kit-Acosta-Borges-Pelinski"], check=True)
os.chdir("hackathon-kit-Acosta-Borges-Pelinski") if os.path.basename(os.getcwd()) != "hackathon-kit-Acosta-Borges-Pelinski" else None
sys.path.insert(0, os.getcwd())

!pip -q install -r requirements.txt

from ic_kit.checkpoints import montar_drive
montar_drive()
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## Datos

In [ ]:
import gdown
gdown.download("https://drive.google.com/uc?id=PEGAR_ID", "train.zip")
!unzip -o -qq train.zip

gdown.download("https://drive.google.com/uc?id=PEGAR_ID_SUBMIT", "only_submit.zip")
!unzip -o -qq only_submit.zip

## Configuración

`CLASES` va en el orden que declara la consigna, que es el que usa el servidor
para indexar la matriz de costos. Pasar esa lista al generador de Keras es
obligatorio: si se omite, Keras ordena las carpetas alfabéticamente y las
etiquetas quedan corridas.

In [ ]:
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from ic_kit import costs, traps
from ic_kit import submit as sub_mod
from ic_kit.bitacora import Bitacora

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)

DIR_TRAIN  = "./train"
DIR_SUBMIT = "./only_submit"

CLASES = ["Pecari_tajacu", "Nasua_nasua", "Harpia_harpyja", "Caiman_latirostris",
          "Phyllomedusa_distincta", "Aspidosperma_polyneuron",
          "Philodendron_bipinnatifidum", "Dicksonia_sellowiana",
          "Aechmea_distichantha", "Ilex_paraguariensis"]

GRUPOS = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])      # reino de cada clase
COSTO  = costs.grouped_cost_matrix(GRUPOS, same_group=2.0, diff_group=5.0)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
MENOR_ES_MEJOR = True

carpetas = sorted(os.listdir(DIR_TRAIN))
assert set(carpetas) == set(CLASES), "carpetas y CLASES no coinciden: %s" % (
    set(carpetas) ^ set(CLASES))
bit = Bitacora()
print("GPU:", tf.config.list_physical_devices("GPU"))
pd.DataFrame(COSTO, index=CLASES, columns=CLASES).astype(int)

## Exploración

Conteo por clase, imágenes rotas y duplicados. Los duplicados importan: si la
misma foto cae en entrenamiento y en validación, la validación da alto y el
ranking no acompaña.

In [ ]:
from ic_kit import vision

rutas, y_all, clases_dir, grupos_hash = vision.build_index(DIR_TRAIN)
orden = {c: i for i, c in enumerate(CLASES)}
y_all = np.array([orden[clases_dir[i]] for i in y_all])       # reindexar al orden de la consigna

rotas = vision.corrupt_images(rutas)
print("imágenes ilegibles:", len(rotas))

cnt = pd.Series(y_all).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar([CLASES[i] for i in cnt.index], cnt.values)
ax.set_ylabel("imágenes"); plt.xticks(rotation=60, ha="right")
ax.set_title("Imágenes por clase")
fig.tight_layout(); plt.show()
print("desbalance máximo: %.1fx" % (cnt.max() / cnt.min()))

Un modelo que acierta mirando sólo color y textura gruesa está
usando el fondo, no el objeto. Si el conjunto de evaluación tiene otros fondos,
eso se desploma.

In [ ]:
acc_atajo = traps.shortcut_scan_images(rutas, y_all, size=16)

## Generadores

Aumentación fuerte de color y recorte, que es la defensa directa contra el
atajo de fondo detectado arriba.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

gen_train = ImageDataGenerator(
    validation_split=0.25, rescale=1./255,
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.25, brightness_range=(0.7, 1.3),
    channel_shift_range=30.0, horizontal_flip=True, fill_mode="reflect")

comun = dict(directory=DIR_TRAIN, class_mode="categorical", classes=CLASES,
             batch_size=BATCH_SIZE, target_size=IMAGE_SIZE, seed=RANDOM_STATE)

train_data = gen_train.flow_from_directory(subset="training", shuffle=True, **comun)
val_data   = ImageDataGenerator(rescale=1./255, validation_split=0.25).flow_from_directory(
    subset="validation", shuffle=False, **comun)

assert list(train_data.class_indices.keys()) == CLASES, "el orden de clases se corrió"
num_classes = len(CLASES)

## Modelo con reanudación

Keras guarda el mejor modelo y también el último de cada época junto con el
número de época. Si la sesión se corta, esta celda retoma desde ahí en vez de
volver a empezar.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Sequential

CKPT = "./CheckPoints"; os.makedirs(CKPT, exist_ok=True)
MEJOR, ULTIMO, ESTADO = f"{CKPT}/mejor.keras", f"{CKPT}/ultimo.keras", f"{CKPT}/estado.json"
EPOCAS = 12

def construir():
    base = MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    m = Sequential([base, GlobalAveragePooling2D(), Dropout(0.3),
                    Dense(num_classes, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
              metrics=["accuracy"])
    return m

epoca_inicial = 0
if os.path.exists(ULTIMO) and os.path.exists(ESTADO):
    modelo = keras.models.load_model(ULTIMO)
    epoca_inicial = json.load(open(ESTADO))["epoca"]
    print("reanudando desde la época", epoca_inicial)
else:
    modelo = construir()

class GuardarEstado(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.save(ULTIMO)
        tmp = ESTADO + ".tmp"
        json.dump({"epoca": epoch + 1}, open(tmp, "w"))
        os.replace(tmp, ESTADO)

callbacks = [
    keras.callbacks.ModelCheckpoint(MEJOR, monitor="val_loss", save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
    GuardarEstado(),
]

hist = modelo.fit(train_data, validation_data=val_data, epochs=EPOCAS,
                  initial_epoch=epoca_inicial, callbacks=callbacks)

Segunda etapa: descongelar la parte alta del backbone con una
tasa de aprendizaje baja. Suele valer varios puntos y es lo que separa un
transfer learning superficial de uno hecho en serio.

In [ ]:
base = modelo.layers[0]
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False

modelo.compile(optimizer=keras.optimizers.Adam(1e-5),
               loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
               metrics=["accuracy"])

hist2 = modelo.fit(train_data, validation_data=val_data, epochs=8, callbacks=callbacks)
modelo = keras.models.load_model(MEJOR)

## Validación y decisión sensible al costo

Se predice sobre **toda** la validación, no sobre un lote: con clases poco
frecuentes, un lote de 256 imágenes puede traer tres ejemplos de una clase y el
número resultante no significa nada.

In [ ]:
val_data.reset()
p_val = modelo.predict(val_data, verbose=1)
y_val = val_data.classes[:len(p_val)]

print("exactitud            :", round(float((p_val.argmax(1) == y_val).mean()), 4))
print(costs.decision_gain(y_val, p_val, COSTO))

pred_val = costs.bayes_decision(p_val, COSTO)
err_reino = float((GRUPOS[pred_val] != GRUPOS[y_val]).mean())
print("errores de reino (los que cuestan 5): %.2f%%" % (100 * err_reino))
print()
print(costs.confusion_cost_report(y_val, pred_val, COSTO, labels=CLASES))

Imágenes que el modelo cree mal etiquetadas, ordenadas por daño a
la métrica. Revisar las primeras a ojo rinde más que otra época de
entrenamiento.

In [ ]:
rutas_val = [os.path.join(DIR_TRAIN, f) for f in val_data.filenames]
orden_sospecha, _ = vision.suspect_labels(p_val, y_val, COSTO, top=12)

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for ax, i in zip(axes.ravel(), orden_sospecha):
    ax.imshow(plt.imread(rutas_val[i])); ax.axis("off")
    ax.set_title("es: %s\nmodelo: %s" % (CLASES[y_val[i]][:18], CLASES[pred_val[i]][:18]),
                 fontsize=8)
fig.tight_layout(); plt.show()

## Predicción con aumentación en test

Promediar la imagen original y su espejo reduce la varianza de la predicción
sin costo de entrenamiento.

In [ ]:
gen_sub = ImageDataGenerator(rescale=1./255)
sub_data = gen_sub.flow_from_directory(DIR_SUBMIT, class_mode=None, shuffle=False,
                                       target_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
sub_data.reset(); p1 = modelo.predict(sub_data, verbose=1)

gen_flip = ImageDataGenerator(rescale=1./255, horizontal_flip=True)
sub_flip = gen_flip.flow_from_directory(DIR_SUBMIT, class_mode=None, shuffle=False,
                                        target_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
sub_flip.reset(); p2 = modelo.predict(sub_flip, verbose=1)

p_sub = (p1 + p2) / 2
pred_sub = costs.bayes_decision(p_sub, COSTO)

## Envío

In [ ]:
nombres = [os.path.basename(f) for f in sub_data.filenames]
s = sub_mod.make_submission(nombres, pred_sub, path="work/submit.csv",
                            id_col="Image", target_col="Predict")
sub_mod.validate(s, allowed_labels=list(range(num_classes)))

log = sub_mod.SubmitLog(lower_is_better=MENOR_ES_MEJOR)
print(log.can_submit())

## Probatorio

In [ ]:
np.save("work/oof.npy", p_val); np.save("work/y.npy", y_val)

from ic_kit.probatorio import generar_notebook
generar_notebook(
    grupo="...", integrantes=["Acosta", "Borges", "Pelinski"],
    desafio="...", metrica="costo promedio por imagen; menor es mejor",
    oof="work/oof.npy", y="work/y.npy", C=COSTO, clases=CLASES)
bit.mostrar()